In [1]:
import os
import sys
import numpy as np
from keras.models import load_model
import cv2
import time
import argparse

sys.path.append("/path/to/file/")
from utils.data_manager import DSManager
from utils.common import shuffle_in_unison
from utils.preprocessor import preprocess_input

caffe_root = '/home/ubuntu/caffe/' ### Change your caffe_root here
import caffe
from caffe import layers as L, params as P
from ltnet import EmotEngine

parser = argparse.ArgumentParser(description="")
parser.add_argument('--weight', '-w', type=str, help='evaluate model')
args = parser.parse_args()
 
def keras_load(m_path):
    model = load_model(m_path)
    return model


def keras_test(model, data):
    correct = 0
    images = data[0]
    actual = data[1]
    sum_time = 0
    for i, image in enumerate(images):
        image = cv2.resize(image, (64, 64))
        # image = preprocess_input(image, True)
        start = time.time()
        image = np.expand_dims(image, 0)
        image = np.expand_dims(image, -1)
        prediction = model.predict(image)
        prediction = np.argmax(prediction)
        sum_time += time.time() - start
        if prediction == actual[i]:
            correct += 1

    accuracy = correct/len(images)
    print("keras latency", (sum_time/(len(images))))

    return accuracy


ds_root = 'data'
# ds_names = ['AffectNet', 'ExpW', 'FER2013', 'KDEF']
ds_names = ['FER2013']
# ds_names = ['FER2013']
m_root = "models"
m_names = []
fer_path = "/home/ubuntu/Repos/face_classification/trained_models/undersampled"
input_shape = (64, 64)
split = .1
type = "bw"
accuracies = []
ds_dict = {}
num_classes = 7

batchsize = 64
src_size = 64
input_data_str = "input"
sys.path.append("/path/to/file/")

/usr/local/lib/python2.7/dist-packages/h5py/__init__.py:36: FutureWarning: Conversion of the second argument of issubdtype from `float` to `np.floating` is deprecated. In future, it will be treated as `np.float64 == np.dtype(float).type`.
  from ._conv import register_converters as _register_converters
Using TensorFlow backend.


ImportError: Traceback (most recent call last):
  File "/usr/local/lib/python2.7/dist-packages/tensorflow/python/pywrap_tensorflow.py", line 58, in <module>
    from tensorflow.python.pywrap_tensorflow_internal import *
  File "/usr/local/lib/python2.7/dist-packages/tensorflow/python/pywrap_tensorflow_internal.py", line 28, in <module>
    _pywrap_tensorflow_internal = swig_import_helper()
  File "/usr/local/lib/python2.7/dist-packages/tensorflow/python/pywrap_tensorflow_internal.py", line 24, in swig_import_helper
    _mod = imp.load_module('_pywrap_tensorflow_internal', fp, pathname, description)
ImportError: libcublas.so.9.0: cannot open shared object file: No such file or directory


Failed to load the native TensorFlow runtime.

See https://www.tensorflow.org/install/install_sources#common_installation_problems

for some common reasons and solutions.  Include the entire stack trace
above this error message when asking for help.

In [ ]:
import sys
# sys.path.append("/home/ubuntu/Repos/fer_tools/utils/data_manager.py")
# print(sys.path)
from utils.data_manager import DSManager
# from utils.data_manager import DSManager

In [ ]:
def eval_validation(model_weights, model_def="res80_fer_deploy.prototxt",
                    device_no=1, batchsize=64, faces=[], labels=[], num_classes=7):

    predictor = EmotEngine.EmotEngine(model_def, model_weights,
                                      device_no=device_no, norm_flag=2,
                                      src_size=(36, 36))
    batchsize = predictor.batchsize
    sumloss = 0.0
    sumacc = 0
    num_images = len(faces)
    batches = float(num_images)/float(batchsize)
    for i in range(math.floor(batches)):
        face_list = faces[i*batchsize:i*batchsize+batchsize]
        label_list = labels[i*batchsize:i*batchsize+batchsize]
        iacc, iloss = predictor.get_emoti_and_loss_from_batch(face_list,
                                                              label_list,
                                                              nclass=num_classes)
        sumloss += iloss*num_images
        sumacc += sum(iacc)
    if batches % math.floor(batches) != 0:
        face_list = faces[i*batchsize:len(faces)]
        label_list = labels[i*batchsize:len(faces)]
        iacc, iloss = predictor.get_emoti_and_loss_from_batch(face_list,
                                                              label_list,
                                                              nclass=num_classes)
        sumloss += iloss*num_images
        sumacc += sum(iacc)
    sumacc = float(sumacc)/float(num_images)
    print("acc = %f" % sumacc)

    return sumacc

In [ ]:
# LOAD DATASETS INTO DICTIONARY
print("Loading datasets...")
for i, ds_name in enumerate(ds_names):
    # load/create csv

    csv_name = str(ds_name)+"_"+str(input_shape[0])+"_"+str(int(split*100))+"_"+type+".csv"
    header = "["+str(ds_name)+"]"
    # TODO: default input for csv_file
    data_loader = DSManager(ds_name,
                            ds_root=ds_root,
                            image_size=input_shape[:2],
                            csv_file=csv_name)
    try:
        csv_file = open(os.path.join(ds_root, csv_name))
    except:
        print(header + " File "+csv_name+" does not exist. Exiting.")
        exit()
    else:
        print(header + " File "+csv_name+" exists. Loading file...")
        data_loader.load_split_CSV()

    # load train, test, val
    data_loader.faces = preprocess_input(data_loader.faces)
    ds_dict[ds_name] = [data_loader.faces,
                        data_loader.emotions,
                        []]
                        # [mini_XCEPTION(input_shape, num_classes)]]

    for j, split_count in enumerate(data_loader.split_counts):
        maximum = sum(split_count)
        if j == 0:
            # val_faces = data_loader.faces[split_count[0]:split_count[0]+split_count[1]]
            # val_labels = data_loader.emotions[split_count[0]:split_count[0]+split_count[1]]
            test_faces = data_loader.faces[split_count[0]+split_count[1]:maximum]
            test_labels = data_loader.emotions[split_count[0]+split_count[1]:maximum]
            lindex = maximum
        else:
            # val_faces = np.vstack((val_faces, data_loader.faces[lindex+split_count[0]:lindex+split_count[0]+split_count[1]]))
            # val_labels = np.vstack((val_labels, data_loader.emotions[lindex+split_count[0]:lindex+split_count[0]+split_count[1]]))
            test_faces = np.vstack((test_faces, data_loader.faces[lindex+split_count[0]+split_count[1]:lindex+maximum]))
            test_labels = np.hstack((test_labels, data_loader.emotions[lindex+split_count[0]+split_count[1]:lindex+maximum]))
            lindex = lindex + maximum

    # val_faces, val_labels = shuffle_in_unison(val_faces, val_labels)
    test_faces, test_labels = shuffle_in_unison(test_faces, test_labels)

    model_weight = "/home/ubuntu/Repos/fer_tools/snapshot/train_HidGT_noU_iter_300000.caffemodel"
    device_no = 0
    print("LOG<<" + model_weight)
    accuracy = eval_validation(model_weight,
                               model_def="ltnet/res80_fer_deploy.prototxt",
                               device_no=device_no, batchsize=64,
                               faces=test_faces, labels=test_labels,
                               num_classes=num_classes)

    del(data_loader)
    # # del(val_faces)
    # # del(val_labels)
    del(test_faces)
    del(test_labels)

print(accuracies)